# 01 - 数据探索与清洗

目标：
1. 加载 Olist 原始 CSV
2. 检查数据质量（缺失值、异常值、类型）
3. 清洗并输出 processed 数据

In [ ]:
import pandas as pd
import numpy as np
import os

RAW = '../data/raw'
PROCESSED = '../data/processed'
os.makedirs(PROCESSED, exist_ok=True)

## 1. 加载数据

In [ ]:
customers = pd.read_csv(f'{RAW}/olist_customers_dataset.csv')
orders = pd.read_csv(f'{RAW}/olist_orders_dataset.csv')
order_items = pd.read_csv(f'{RAW}/olist_order_items_dataset.csv')
payments = pd.read_csv(f'{RAW}/olist_order_payments_dataset.csv')
reviews = pd.read_csv(f'{RAW}/olist_order_reviews_dataset.csv')
products = pd.read_csv(f'{RAW}/olist_products_dataset.csv')
sellers = pd.read_csv(f'{RAW}/olist_sellers_dataset.csv')
category_trans = pd.read_csv(f'{RAW}/product_category_name_translation.csv')

print('=== 数据集行数 ===')
for name, df in [('customers', customers), ('orders', orders), ('order_items', order_items),
                  ('payments', payments), ('reviews', reviews), ('products', products),
                  ('sellers', sellers), ('category_trans', category_trans)]:
    print(f'{name:20s}: {len(df):>10,} rows')

## 2. 数据概览

In [ ]:
# 订单时间范围
orders['order_purchase_timestamp'] = pd.to_datetime(orders['order_purchase_timestamp'])
print(f'订单时间范围: {orders.order_purchase_timestamp.min()} ~ {orders.order_purchase_timestamp.max()}')
print(f'订单状态分布:')
print(orders['order_status'].value_counts())

In [ ]:
# 缺失值检查
print('=== 缺失值 ===')
for name, df in [('orders', orders), ('order_items', order_items), ('payments', payments), ('products', products)]:
    missing = df.isnull().sum()
    missing = missing[missing > 0]
    if len(missing) > 0:
        print(f'\n{name}:')
        print(missing)
    else:
        print(f'\n{name}: 无缺失')

## 3. 数据清洗

In [ ]:
# 只保留有效订单（非取消、非不可用）
valid_orders = orders[~orders['order_status'].isin(['canceled', 'unavailable'])].copy()
print(f'有效订单: {len(valid_orders):,} / {len(orders):,} ({len(valid_orders)/len(orders)*100:.1f}%)')

# 订单时间转换
time_cols = ['order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date',
             'order_delivered_customer_date', 'order_estimated_delivery_date']
for col in time_cols:
    valid_orders[col] = pd.to_datetime(valid_orders[col])

# 支付金额检查
print(f'\n支付金额统计:')
print(payments['payment_value'].describe())

# 过滤异常支付（< 0 或 > 10000）
payments_clean = payments[(payments['payment_value'] > 0) & (payments['payment_value'] < 10000)].copy()
print(f'\n清洗后支付记录: {len(payments_clean):,} / {len(payments):,}')

## 4. 构建宽表

In [ ]:
# 订单宽表
order_wide = (
    valid_orders
    .merge(customers, on='customer_id', how='left')
    .merge(order_items, on='order_id', how='left')
    .merge(products, on='product_id', how='left')
    .merge(category_trans, on='product_category_name', how='left')
    .merge(payments_clean.groupby('order_id').agg(
        payment_value=('payment_value', 'sum'),
        payment_type=('payment_type', 'first'),
        payment_installments=('payment_installments', 'max')
    ).reset_index(), on='order_id', how='left')
    .merge(reviews.groupby('order_id')['review_score'].mean().reset_index(), on='order_id', how='left')
)

print(f'宽表行数: {len(order_wide):,}')
print(f'列数: {len(order_wide.columns)}')
print(f'\n列名: {list(order_wide.columns)}')

## 5. 保存清洗数据

In [ ]:
order_wide.to_csv(f'{PROCESSED}/order_wide.csv', index=False)
customers.to_csv(f'{PROCESSED}/customers.csv', index=False)
valid_orders.to_csv(f'{PROCESSED}/orders_clean.csv', index=False)

# 客户首单表
first_orders = (
    valid_orders
    .merge(customers[['customer_id', 'customer_unique_id']], on='customer_id')
    .groupby('customer_unique_id')
    .agg(first_order_date=('order_purchase_timestamp', 'min'))
    .reset_index()
)
first_orders.to_csv(f'{PROCESSED}/customer_first_orders.csv', index=False)

print('已保存:')
for f in os.listdir(PROCESSED):
    size = os.path.getsize(f'{PROCESSED}/{f}') / 1024 / 1024
    print(f'  {f}: {size:.1f} MB')